# Objective 1: TF-IDF, PCA and monthly topic discovery

PCA is fitted to cleaned TF-IDF features. Explained variance is measured **before row L2 normalization** and is not semantic accuracy.
Exploratory comparisons used 50, 100, 200 and 400 components with the same corpus and vocabulary. The selected configuration uses **PCA100, row L2 normalization and K-means with 18 clusters**.

The frozen sample contains 10,000 comments from January through October 2026; October is partial. The proposal specified PCA50, so report this requested methodological change alongside the original baseline.
The notebook reuses the verified model by default and presents six cases: **C2, C5, C8, C9, C11 and C17**, considering semantics, size, project relevance and trend robustness.
The heatmap hides only C7; all denominators, assignments and full exports still include all 18 clusters.


## 1. Environment, frozen model and label binding

The default `REUSE_SAVED_MODEL=True` mode verifies and reuses the previously analysed PCA100/L2/k=18 model. It does not fit TF-IDF, PCA or K-means again.
`SAVED_MODEL_DIR` points to an ignored local result folder. Pipeline, assignment and input SHA-256 values must match the review binding; missing or changed files stop reuse.

The next cell embeds 18 English topic suggestions, interpretation confidence and limitations. These are **assistant interpretations pending human validation**, not gold labels or model-generated confidence scores.
Use `TOPIC_LABELS` only after actual human review. Turning reuse off fits a new, unnamed model; old labels are not transferred by cluster number.
Network acquisition is disabled. Restart the kernel and run from the top after editing modules.

GitHub distribution excludes Reddit data, model files and notebook outputs. A fresh clone without local data skips modelling rather than generating substitute data. Restore permitted local files to reproduce the frozen report; see `scripts/README_objective1.md`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import subprocess
import sys
import os
import joblib
from time import perf_counter

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '2')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd().resolve()
if not (ROOT / 'functions').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'functions' / 'topic_data.py').exists():
    raise RuntimeError('Start this notebook from the project root or notebook directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from functions.reddit_collection import read_comments
from functions.preprocessing import cleanup_text
from functions.topic_data import month_windows, export_topic_run
from functions.topics import (
    verified_coverage, vectorize_comments, reduce_topics, topic_coordinates, original_space_diagnostics, evaluate_clusters,
    topic_profiles as build_topic_profiles, initialization_stability,
)

from functions.topic_reporting import topic_trend_diagnostics

SEED = 42
YEAR = 2026
PER_MONTH = 1000
MIN_WORDS = 3
AS_OF = pd.Timestamp.now(tz='UTC').floor('s').isoformat()
DATA_ROOT = ROOT / 'dataset' / 'reddit' / 'topics_2026'

# Select an existing prepared run; None searches local run records.
RUN_DIR = ROOT / 'dataset/reddit/topics_2026/time_stratified_20261028/prepared'
RAW_EXPORT_PATH = None  # Path to a local historical CSV with the four required columns.
RUN_ARCHIVE_DOWNLOAD = False
RUN_TOPIC_MODEL = True
SELECTED_K = 18  # Use the reviewed k; None supplies a provisional candidate when fitting.
REUSE_SAVED_MODEL = True  # Reuse the verified named model without refitting.
SAVED_MODEL_DIR = RUN_DIR / 'topics_pca100_l2_20261029T105658930366Z'
TOPIC_LABELS = {}  # After human review: {cluster_id: 'Topic name'}.
SAVE_TOPIC_OUTPUTS = True
LAYOFF_CSV = None  # Optional CSV with month,laid_off columns.
LAYOFF_CLUSTER_IDS = []  # Choose from content, not by matching an external curve.

versions = {name: importlib.metadata.version(name)
            for name in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']}
print('Python:', sys.version.split()[0])
print(versions)
print('UTC cutoff for a new acquisition:', AS_OF)
display(pd.DataFrame(month_windows(YEAR, AS_OF)))

PCA_COMPONENTS = 100
PCA_L2_NORMALIZE = True
BASELINE_COMPONENTS = 50
K_VALUES = [8, 10, 12, 15, 18, 20]
SILHOUETTE_SAMPLE = 1500
N_INIT = 30
THREADS = 2
EXCLUDE_MODERATION_NOTICES = True
# English stop words include no/not/nor; retain off for laid off.
# Early PCA was dominated by fillers; raw text and classifier inputs are unchanged.
EXTRA_TOPIC_STOP_WORDS = [
    'just', 'like', 'really', 'actually', 'probably', 'maybe', 'yes', 'yeah',
    'lol', 'sounds', 'know', 'think', 'people', 'good', 'need', 'want',
    'going', 'got', 'did', 'does', 'said', 'say', 'don', 've', 'll', 'gif', 'giphy',
]
KEEP_TOPIC_WORDS = ['off']
features_ready = False
final_ready = False
figures = {}

# Bind suggestions to reviewed weights and assignments, not new cluster IDs.
REVIEW_BINDING = {
    'pipeline': '6be0804458701fc1d1dce6684fb30e9dbd7c167a2e8af359cb8ceadce5e548ef',
    'assignments': '10bd31d6f387cb4a8889ac529b8c37f349633406856a7307b1edff4fa05a5e79',
    'input': '62111327682df76d2d44899c0f28cceddbe51b3800d871d684cdf29b34ea1fc3',
}
saved_bundle = None
review_binding_verified = False

CASE_STUDY_IDS = [2, 5, 8, 9, 11, 17]
HEATMAP_HIDDEN_CLUSTERS = [7]  # Hide the plotted row only; all denominators still include C7.


In [ ]:
TOPIC_REVIEW = [{'cluster': 0,
  'topic_label': 'Work time and leave (mixed)',
  'semantic_confidence': 'Low-medium',
  'employment_topic': True,
  'interpretation_limit': 'Time includes leave, wasted time and life stages; not solely leave policy.',
  'trend_judgment': 'Fluctuates without a clear sustained direction.'},
 {'cluster': 1,
  'topic_label': 'Work arrangements and job advice (broad)',
  'semantic_confidence': 'Low',
  'employment_topic': True,
  'interpretation_limit': 'Generic work wording mixes industry advice, visas and employment verification.',
  'trend_judgment': 'No clear trend; equal-post weighting changes the direction.'},
 {'cluster': 2,
  'topic_label': 'Manager/HR communication and resignation notice',
  'semantic_confidence': 'Medium',
  'employment_topic': True,
  'interpretation_limit': 'Communication, recruitment management, leave and resignation notice are mixed; '
                          'not solely conflict.',
  'trend_judgment': 'Strongest upward display candidate, with mixed semantics and descriptive interpretation '
                    'only.'},
 {'cluster': 3,
  'topic_label': 'Past experiences and elapsed years',
  'semantic_confidence': 'Low',
  'employment_topic': False,
  'interpretation_limit': 'Years ago is an expression pattern, not necessarily qualification requirements.',
  'trend_judgment': 'No clear topic trend; elapsed-year wording is not a qualifications indicator.'},
 {'cluster': 4,
  'topic_label': 'Job choices and labour-market discussion (broad)',
  'semantic_confidence': 'Low-medium',
  'employment_topic': True,
  'interpretation_limit': 'Generic job wording is not a direct labour-market deterioration indicator.',
  'trend_judgment': 'No clear sustained trend; broad job discussion is not an economic indicator.'},
 {'cluster': 5,
  'topic_label': 'Pay, pay cuts and payment issues',
  'semantic_confidence': 'Medium-high',
  'employment_topic': True,
  'interpretation_limit': 'Payment-related meaning is relatively consistent but includes rent, scams and '
                          'general payments.',
  'trend_judgment': 'Broadly stable; useful as a stable-topic reference.'},
 {'cluster': 6,
  'topic_label': 'Company evaluations and employment processes',
  'semantic_confidence': 'Low-medium',
  'employment_topic': True,
  'interpretation_limit': 'Company wording mixes checks, hiring, evaluations and advice.',
  'trend_judgment': 'Fluctuates without a clear sustained direction.'},
 {'cluster': 7,
  'topic_label': 'General advice and responses (mixed)',
  'semantic_confidence': 'Low',
  'employment_topic': False,
  'interpretation_limit': 'Large residual cluster; do not treat it as one employment topic.',
  'trend_judgment': 'Large and broadly stable; unsuitable for a single-topic interpretation.'},
 {'cluster': 8,
  'topic_label': 'Job applications and opportunities',
  'semantic_confidence': 'Medium',
  'employment_topic': True,
  'interpretation_limit': 'Applications, job sources and opportunities coexist with generic jobs wording.',
  'trend_judgment': 'Declining signal with intermediate rebounds; moderate-to-weak evidence.'},
 {'cluster': 9,
  'topic_label': 'Job changes, new jobs and onboarding',
  'semantic_confidence': 'High',
  'employment_topic': True,
  'interpretation_limit': 'New work, resignation timing, offers and job-search advice are relatively '
                          'coherent.',
  'trend_judgment': 'Lower April-May and higher August-September; not a continuous increase.'},
 {'cluster': 10,
  'topic_label': 'Judgment and agreement wording',
  'semantic_confidence': 'Low',
  'employment_topic': False,
  'interpretation_limit': 'Right is not an independent employment topic.',
  'trend_judgment': 'Discourse pattern; do not interpret as an employment-topic trend.'},
 {'cluster': 11,
  'topic_label': 'Experience, education and qualifications',
  'semantic_confidence': 'Medium-high',
  'employment_topic': True,
  'interpretation_limit': 'Experience and education are relatively coherent, with some generic '
                          'personal-experience references.',
  'trend_judgment': 'Declining signal, with a June rebound and model-boundary uncertainty.'},
 {'cluster': 12,
  'topic_label': 'Miscellaneous short and emotional replies',
  'semantic_confidence': 'Low',
  'employment_topic': False,
  'interpretation_limit': 'A few terms such as labour, income or Trump do not establish a political-economic '
                          'theme.',
  'trend_judgment': 'Residual replies; do not impose a single-topic interpretation.'},
 {'cluster': 13,
  'topic_label': 'Questions and requests for advice',
  'semantic_confidence': 'Low',
  'employment_topic': False,
  'interpretation_limit': 'Ask groups questions across contexts; not solely interview questions.',
  'trend_judgment': 'Some numerical increase, but mainly a question/answer expression pattern.'},
 {'cluster': 14,
  'topic_label': 'Workdays, hours and schedules (mixed)',
  'semantic_confidence': 'Low-medium',
  'employment_topic': True,
  'interpretation_limit': 'Day also refers to account age, cake day and reminders; not a pure work-hours '
                          'cluster.',
  'trend_judgment': 'Some numerical increase, but mixed day meanings prevent a clear topic claim.'},
 {'cluster': 15,
  'topic_label': 'Responses about the original poster',
  'semantic_confidence': 'High (discourse form)',
  'employment_topic': False,
  'interpretation_limit': 'OP indicates discourse form, not an employment theme.',
  'trend_judgment': 'Original-poster response form; not an employment-topic trend.'},
 {'cluster': 16,
  'topic_label': 'Work and life choices (broad)',
  'semantic_confidence': 'Low-medium',
  'employment_topic': True,
  'interpretation_limit': 'Life includes outlook, health and career choices; not solely work-life balance.',
  'trend_judgment': 'Numerical decline with mixed semantics; do not infer improving life pressure.'},
 {'cluster': 17,
  'topic_label': 'AI use, generated content and job substitution',
  'semantic_confidence': 'High',
  'employment_topic': True,
  'interpretation_limit': 'Content identification, AI use and substitution coexist; not solely unemployment '
                          'anxiety.',
  'trend_judgment': 'Higher February-May and generally lower after June; not a continuous decline from '
                    'January.'}]

CASE_STUDY_CONTEXT = {2: {'semantic_evidence': 'Manager, HR and notice terms support communication/resignation discussion; '
                          'generic tell wording keeps the rating at Medium.',
     'project_relevance': 'Hiring, management communication and resignation are relevant to workplace '
                          'interaction and later topic-level toxicity analysis.',
     'robustness_assessment': 'Comment shares: 4.98% to 6.85%; equal-post shares: 5.95% to 7.23%. Direction '
                              'agrees, but mixed meaning does not establish increasing conflict.'},
 5: {'semantic_evidence': 'Pay, pay cut, bills and rate support salary/payment discussion, with some general '
                          'payments.',
     'project_relevance': 'Compensation and economic-pressure discussion provide an employment-related '
                          'stable reference.',
     'robustness_assessment': 'Comment shares: 3.18% to 3.17%; equal-post: 3.00% to 3.03%. Both are near '
                              'stable; some months have larger single-post influence.'},
 8: {'semantic_evidence': 'Apply, applying, jobs and resume support applications and opportunities, with '
                          'generic jobs references.',
     'project_relevance': 'Applications, opportunities and recruitment channels directly concern the core '
                          'r/jobs discussion.',
     'robustness_assessment': 'Comment shares: 4.84% to 3.82%; equal-post: 5.49% to 3.90%. Decline agrees, '
                              'with intermediate rebounds.'},
 9: {'semantic_evidence': 'New job, start and quit support job changes, onboarding and waiting for a '
                          'confirmed offer before resigning.',
     'project_relevance': 'Work transitions and joining a new employer are relevant; discussion shares are '
                          'not actual switching counts.',
     'robustness_assessment': 'Lower April-May and higher August-September; quarterly change from 2.95% to '
                              '3.30% is small, and peak month depends on post weighting.'},
 11: {'semantic_evidence': 'Experience, degree and entry level support qualifications, education and '
                           'resumes, with some generic experience references.',
      'project_relevance': 'Experience requirements and education/job fit are relevant; do not infer '
                           'employers lowered qualification requirements.',
      'robustness_assessment': 'Comment shares: 3.86% to 2.79%; equal-post: 4.30% to 3.57%. Decline agrees, '
                               'with a June rebound rather than a continuous fall.'},
 17: {'semantic_evidence': 'AI, generated, replace and slop support tools, content identification and '
                           'substitution, not solely job-loss anxiety.',
      'project_relevance': 'AI use and labour substitution matter, but AI-content identification prevents '
                           'use as a direct layoffs indicator.',
      'robustness_assessment': 'Comment shares: 2.57% to 1.55%; equal-post: 2.45% to 1.68%. Both show '
                               'phase-dependent decline; only about 10-33 comments per month.'}}


## 2. Historical source and time-stratified sampling

Shared CSV reading and text cleaning are reused. `scripts/collect_monthly_topics.py` retrieves historical comments; the existing `collect_reddit.py collect` reads recent listings and cannot backfill historical months.

Each UTC month is divided into 20 equal-duration strata. Seeded random anchors define bounded candidate batches, with continuation or wraparound when necessary. Cleaning precedes sampling 50 eligible comments without replacement per stratum.
Requests are rate-limited and checkpointed. An interrupted run can resume from the same work directory.

This is random-anchor time-stratified candidate sampling, not enumeration or a simple random sample of the complete monthly population. `source_window_complete=False` records that design limitation. Archive coverage, unknown bots and unverified language remain limitations.
To acquire a new corpus, enable `RUN_ARCHIVE_DOWNLOAD` and use a new run folder. Local imports require `comment_id,text,created_utc,subreddit`, with UTC Unix seconds.


In [ ]:
if RAW_EXPORT_PATH is not None and RUN_ARCHIVE_DOWNLOAD:
    raise ValueError('Choose a local export or an archive download for this run.')

if RAW_EXPORT_PATH is not None or RUN_ARCHIVE_DOWNLOAD:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    RUN_DIR = DATA_ROOT / ('import_' + stamp if RAW_EXPORT_PATH is not None else 'archive_' + stamp)
    if RAW_EXPORT_PATH is not None:
        raw_path = Path(RAW_EXPORT_PATH)
        records = read_comments(raw_path)
        export_topic_run(
            RUN_DIR, records, YEAR, AS_OF,
            {'method': 'local_csv', 'file': raw_path.name,
             'sha256': hashlib.sha256(raw_path.read_bytes()).hexdigest(),
             'coverage': 'not independently verified'},
            per_month=PER_MONTH, seed=SEED, min_words=MIN_WORDS,
        )
    else:
        work_dir = RUN_DIR
        result = subprocess.run(
            [sys.executable, str(ROOT / 'scripts' / 'collect_monthly_topics.py'),
             '--work-dir', str(work_dir), '--year', str(YEAR),
             '--per-month', str(PER_MONTH), '--seed', str(SEED),
             '--min-words', str(MIN_WORDS), '--as-of', AS_OF],
            cwd=ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace',
        )
        RUN_DIR = work_dir / 'prepared'
        print(result.stdout)
        if result.returncode:
            print('Acquisition did not finish. Inspect its status report:', result.stderr)

if RUN_DIR is None:
    runs = [path for path in DATA_ROOT.glob('*') if path.is_dir()]
    RUN_DIR = max(runs, key=lambda path: path.stat().st_mtime) if runs else None
elif not Path(RUN_DIR).is_absolute():
    RUN_DIR = ROOT / RUN_DIR
print('Selected run:', RUN_DIR)

## 3. Verify acquisition and coverage

`source_error` and `not_attempted` represent missing data, not zero-comment months. Source-window completeness concerns the archive response, not complete Reddit coverage.
Use the selected manifest's cutoff rather than today's date.

If spreadsheet software changed only month labels or boolean casing, the loader may restore metadata in memory only when reconstructed bytes match the original manifest hash. Source files remain unchanged. Changed comment content or other unmatched hashes stop execution.


In [ ]:
corpus = pd.DataFrame()
coverage = pd.DataFrame()
manifest = {}
acquisition = {}
if RUN_DIR is not None:
    RUN_DIR = Path(RUN_DIR)
    status_path = RUN_DIR / 'acquisition_status.json'
    if not status_path.exists():
        status_path = RUN_DIR.parent / 'acquisition_status.json'
    if status_path.exists():
        acquisition = json.loads(status_path.read_text(encoding='utf-8'))
        if 'months' in acquisition:
            display(pd.DataFrame(acquisition['months']))
        elif 'strata' in acquisition:
            progress = pd.DataFrame(acquisition['strata'])
            if not progress.empty:
                display(progress.groupby('month').agg(
                    completed_strata=('complete', 'sum'),
                    eligible_candidates=('eligible_candidates', 'sum'),
                    target_samples=('quota', 'sum'),
                ))
        print('Retrieval status:', acquisition['status'])
    manifest_path = RUN_DIR / 'sampling_manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        for name, expected_hash in manifest['sha256'].items():
            if name == 'monthly_coverage.csv':
                coverage = verified_coverage(RUN_DIR / name, expected_hash)
                continue
            actual_hash = hashlib.sha256((RUN_DIR / name).read_bytes()).hexdigest()
            if actual_hash != expected_hash:
                raise ValueError(f'File changed since preparation: {name}')
        if (manifest['year'], manifest['target_per_month'], manifest['seed'], manifest['min_words']) != (
            YEAR, PER_MONTH, SEED, MIN_WORDS
        ):
            raise ValueError('Selected run uses different parameters. Select or prepare the intended run.')
        print('Run cutoff:', manifest['as_of_utc'])
        corpus = pd.read_csv(RUN_DIR / 'comments_clean.csv', dtype={'comment_id': str}, keep_default_na=False)
        display(coverage)
        display(pd.DataFrame([manifest['counts']]))
        if not corpus.empty:
            assert corpus['comment_id'].is_unique
            assert corpus['text'].str.strip().ne('').all()
            corpus['created_at'] = pd.to_datetime(corpus['created_at'], utc=True)
            assert corpus.groupby('month').size().le(PER_MONTH).all()
            assert len(corpus) == manifest['counts']['selected']
            if manifest.get('strata_per_month') == 20:
                strata_counts = corpus.groupby(['month', 'sampling_stratum']).size()
                assert strata_counts.eq(PER_MONTH // 20).all()
                assert corpus.groupby('month')['sampling_stratum'].nunique().eq(20).all()
                assert coverage['selected'].eq(PER_MONTH).all()

if corpus.empty:
    print('No prepared historical corpus is available. Modelling cells will skip; no substitute data is used.')

## 4. Cleaning and provenance audit

1. Check comment IDs, r/jobs membership and UTC timestamps; exclude out-of-window or future records.
2. Exclude missing bodies, `[deleted]`, `[removed]`, known bots and explicitly labelled non-English records.
3. Deduplicate by ID; conflicting snapshots stop preparation. Retain identical text from different IDs and flag it.
4. Preserve raw `text`; decode HTML, lowercase and reuse shared `cleanup_text` to create `text_topic`.
5. Require at least three alphabetic words by default; record every exclusion.
6. Sample 50 eligible candidates without replacement from each of 20 monthly strata, seed 42. Report shortfalls rather than duplicate rows.

Missing language metadata remains unverified. Inspect random comments for language, templates and cleaning damage. Do not remove comments merely because they are difficult to cluster. Toxicity labels and annotation notes are not topic features.


In [ ]:
if manifest:
    audit = pd.read_csv(RUN_DIR / 'cleaning_audit.csv')
    excluded = pd.read_csv(RUN_DIR / 'excluded_rows.csv', keep_default_na=False)
    display(audit)
    if not excluded.empty:
        display(excluded.groupby(['month', 'exclusion_reason'], dropna=False).size().rename('n_rows').reset_index())
    assert int(audit['n_rows'].sum()) == manifest['counts']['raw']
    assert manifest['counts']['eligible'] + manifest['counts']['excluded'] == manifest['counts']['raw']
if not corpus.empty:
    display(corpus[['comment_id', 'month', 'text', 'text_topic', 'word_count',
                    'duplicate_text', 'language_status']].sample(min(10, len(corpus)), random_state=SEED))
    print('Review these random examples for language, bot/template text and over-cleaning.')

## 5. Sample coverage and comment length

These charts describe the collected sample. Fixed monthly quotas support topic-share comparisons, not estimates of subreddit volume. Missing months are not filled with zero activity; partial months remain explicit.
Inspect concentration within posts because comments from the same thread are dependent. Later equal-post sensitivity checks address part of that dependence.


In [ ]:
if not corpus.empty:
    monthly_summary = corpus.groupby('month').agg(n_comments=('comment_id', 'size'),
                                                  n_posts=('link_id', 'nunique'))
    display(monthly_summary)
    display(corpus['link_id'].replace('', np.nan).value_counts().head(10).rename('n_comments').to_frame())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    monthly_summary['n_comments'].plot.bar(ax=axes[0], title='Sample coverage per UTC month')
    axes[0].set_ylabel('Sampled comments (not subreddit volume)')
    corpus['word_count'].plot.hist(bins=30, ax=axes[1], title='Cleaned word counts')
    fig.tight_layout()
    plt.show()

In [ ]:
saved_bundle = None
review_binding_verified = False
if RUN_TOPIC_MODEL and not corpus.empty and REUSE_SAVED_MODEL:
    source_paths = {
        'pipeline': SAVED_MODEL_DIR / 'topic_pipeline.joblib',
        'assignments': SAVED_MODEL_DIR / 'comment_topics.csv',
        'input': RUN_DIR / 'comments_clean.csv',
    }
    for kind, target in source_paths.items():
        if not target.exists() or hashlib.sha256(target.read_bytes()).hexdigest() != REVIEW_BINDING[kind]:
            raise ValueError(f'Reviewed-model binding failed: {kind}. Do not transfer topic names to another model.')
    saved_parameters = json.loads((SAVED_MODEL_DIR / 'run_parameters.json').read_text(encoding='utf-8'))
    for name, expected in saved_parameters['output_sha256'].items():
        if hashlib.sha256((SAVED_MODEL_DIR / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f'Saved result changed: {name}')
    if (PCA_COMPONENTS, PCA_L2_NORMALIZE, SELECTED_K, N_INIT) != (
        saved_parameters['pca_components'], saved_parameters['pca_l2_normalize'],
        saved_parameters['selected_k'], saved_parameters['n_init'],
    ):
        raise ValueError('Reuse mode requires the frozen configuration. Disable reuse to fit a new, unnamed model.')
    saved_bundle = joblib.load(source_paths['pipeline'])
    saved_assignments = pd.read_csv(source_paths['assignments'], dtype={'comment_id': str}, keep_default_na=False)
    review_binding_verified = True
    print('Verified frozen PCA100 / k=18 model. Reporting only; no PCA or K-means refit.')


## 6. TF-IDF features

The retrospective corpus uses a shared unigram/bigram vocabulary: `min_df=3`, `max_df=0.90`, at most 10,000 features, sublinear TF and row L2 normalization. Only `text_topic` enters the model.

Early exploratory fits were dominated by moderation templates and conversational fillers. The documented topic-only configuration therefore excludes explicit moderation-notice phrase combinations and adds listed conversational stop words, retaining `off` for `laid off`.
This does not change raw text or toxicity-classifier preprocessing and does not claim to identify every bot.

IDs excluded as `moderation_notice` or `zero_tfidf` remain in exports. Reuse mode loads the frozen vectorizer and original diagnostics without refitting it.


In [ ]:
if RUN_TOPIC_MODEL and not corpus.empty:
    if saved_bundle is not None:
        print('Frozen TF-IDF reused without fitting; diagnostics below describe the original feature run.')
        vectorizer = saved_bundle['vectorizer']
        assigned_ids = saved_assignments.loc[saved_assignments.topic_status.eq('assigned'), 'comment_id']
        frozen_frame = corpus.loc[corpus.comment_id.isin(assigned_ids)].reset_index(drop=True).copy()
        excluded_frame = saved_assignments.loc[~saved_assignments.topic_status.eq('assigned'),
                                                 ['comment_id', 'month', 'topic_status']].rename(columns={'topic_status': 'reason'})
        feature_result = {
            'vectorizer': vectorizer, 'X': vectorizer.transform(frozen_frame.text_topic),
            'model_df': frozen_frame, 'excluded': excluded_frame,
            'stop_words': saved_parameters['tfidf']['stop_words'],
            'diagnostics': saved_parameters['feature_diagnostics'],
            'top_terms': pd.DataFrame(),
        }
    else:
        feature_result = vectorize_comments(
            corpus, min_df=3, max_df=0.90, max_features=10000,
            extra_stop_words=EXTRA_TOPIC_STOP_WORDS, keep_words=KEEP_TOPIC_WORDS,
            exclude_moderation=EXCLUDE_MODERATION_NOTICES,
        )
    vectorizer = feature_result['vectorizer']
    X = feature_result['X']
    model_df = feature_result['model_df']
    feature_excluded = feature_result['excluded']
    stop_words = feature_result['stop_words']
    feature_diagnostics = feature_result['diagnostics']
    display(pd.DataFrame([feature_diagnostics]))
    display(feature_excluded.groupby('reason').size().rename('n_comments').to_frame())
    display(feature_result['top_terms'])
    assert len(model_df) + len(feature_excluded) == len(corpus)
    assert model_df['comment_id'].is_unique
    print('Sparse TF-IDF shape:', X.shape, '; no dense conversion')
else:
    print('No modelling requested or no corpus available.')


## 7. PCA dimension and projected-vector normalization

The selected dimension balances retained information and topic diagnostics; 90% is not a required threshold. True centered ARPACK PCA is used with `whiten=False`.
The corpus, vocabulary and cleaning remain fixed. Do not inflate retained variance by deleting difficult comments or features.

Keep the raw projection `Z_pca` for PCA variance and use the configured row normalization to produce clustering coordinates `Z`. Normalization changes the distance geometry and is not exactly spherical K-means.
Original TF-IDF cosine diagnostics use a fixed sample of 1,500 rows to provide a common reference across representations. Reuse mode transforms with the saved PCA and normalizer.


In [ ]:
features_ready = False
final_ready = False
if RUN_TOPIC_MODEL and not corpus.empty:
    component_count = min(PCA_COMPONENTS, min(X.shape) - 1)
    if saved_bundle is not None:
        pca = saved_bundle['pca']
        normalizer = saved_bundle['normalizer']
        Z_pca = pca.transform(X)
        Z = normalizer.transform(Z_pca) if normalizer is not None else Z_pca
        pca_variance = pd.read_csv(SAVED_MODEL_DIR / 'pca_variance.csv')
        pca_seconds = 0.0  # Reused here; original fit time is in the frozen run_parameters.json.
    else:
        pca, Z_pca, pca_variance, pca_seconds = reduce_topics(
            X, n_components=component_count, seed=SEED, threads=THREADS,
        )
        Z, normalizer = topic_coordinates(Z_pca, PCA_L2_NORMALIZE)
    actual_components = int(pca.n_components_)
    retained = float(pca_variance.cumulative_variance.iloc[-1])
    baseline_n = min(BASELINE_COMPONENTS, actual_components)
    pca_comparison = pd.DataFrame([
        {'configuration': f'First {baseline_n} components', 'components': baseline_n,
         'retained_variance': float(pca_variance.cumulative_variance.iloc[baseline_n - 1])},
        {'configuration': 'Selected dimension', 'components': actual_components,
         'retained_variance': retained},
    ])
    display(pca_comparison)
    print(f'PCA shape {Z_pca.shape}; fit {pca_seconds:.2f}s; row L2 normalization: {PCA_L2_NORMALIZE}')
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(pca_variance.component, pca_variance.cumulative_variance)
    ax.axvline(baseline_n, linestyle=':', color='grey', label='50-component reference')
    ax.set(xlabel='Components', ylabel='Cumulative variance before normalization', title='Centered PCA retained variance')
    ax.legend()
    fig.tight_layout()
    figures['pca_variance'] = fig
    plt.show()
    features_ready = True
else:
    print('No modelling requested or no prepared corpus available.')


## 8. K diagnostics in the selected representation

Candidates share the same coordinates, K-means++ initialization, seed 42, 30 initializations and fixed silhouette subset. Reuse mode reads the frozen candidate table.
PCA-space inertia and silhouette across different dimensions or normalization settings are not direct quality rankings. Also examine original-space diagnostics, cluster sizes and representative text.


In [ ]:
if features_ready:
    if saved_bundle is not None:
        k_results = pd.read_csv(SAVED_MODEL_DIR / 'k_selection.csv')
        k_models = {SELECTED_K: saved_bundle['kmeans']}
        eval_indices = np.sort(np.random.default_rng(SEED).choice(
            len(Z), min(SILHOUETTE_SAMPLE, len(Z)), replace=False))
    else:
        k_results, k_models, eval_indices = evaluate_clusters(
            Z, ks=K_VALUES, seed=SEED, sample_size=SILHOUETTE_SAMPLE,
            n_init=N_INIT, threads=THREADS,
        )
    display(k_results)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    k_results.plot(x='k', y='inertia', marker='o', legend=False, ax=axes[0], title='Elbow: PCA-space inertia')
    k_results.plot(x='k', y='silhouette', marker='o', legend=False, ax=axes[1], title='Fixed-subset silhouette')
    k_results.plot(x='k', y='largest_cluster_share', marker='o', legend=False, ax=axes[2], title='Largest cluster share')
    axes[2].axhline(0.8, color='grey', linestyle='--', label='80% diagnostic marker')
    axes[2].legend()
    for ax in axes:
        ax.set_xticks(K_VALUES)
    fig.tight_layout()
    figures['k_selection'] = fig
    plt.show()


## 9. Selected model, ambiguity and stability

The default k comes from exploratory comparisons and text inspection. When fitting a new model with `SELECTED_K=None`, size-screened silhouette supplies a provisional candidate requiring review.
Cluster balance does not establish semantic coherence. ARI across seeds 7 and 21 measures initialization sensitivity, not accuracy.
Original TF-IDF cosine silhouette and top-term Jaccard overlap are internal diagnostics, not human semantic validation. Relative nearest/second-nearest centroid margins flag ambiguity and are not probabilities.


In [ ]:
if features_ready:
    feasible = k_results.loc[
        k_results.silhouette.notna() & k_results.actual_clusters.eq(k_results.k)
        & k_results.largest_cluster_share.le(0.80)
        & k_results.smallest_cluster_share.ge(0.005)
    ]
    if feasible.empty:
        print('Warning: no candidate passed the exploratory cluster-size checks. Review before interpreting topics.')
        feasible = k_results.loc[k_results.silhouette.notna() & k_results.actual_clusters.eq(k_results.k)]
    if feasible.empty:
        raise ValueError('No non-degenerate k candidate is available.')
    recommended_k = int(feasible.sort_values(['silhouette', 'k'], ascending=[False, True]).iloc[0].k)
    if SELECTED_K is None:
        SELECTED_K = recommended_k
        print('Provisional k in the new PCA space:', SELECTED_K, '; not a reviewed optimal solution')
    if SELECTED_K not in k_models:
        raise ValueError('SELECTED_K must be among the evaluated k values.')
    final_model = k_models[SELECTED_K]
    if len(np.unique(final_model.labels_)) != SELECTED_K:
        raise ValueError('The requested k produced empty clusters; inspect diagnostics.')
    selected_diagnostics = k_results.loc[k_results.k.eq(SELECTED_K)].copy()
    display(selected_diagnostics)
    if saved_bundle is not None:
        expected_labels = pd.to_numeric(saved_assignments.set_index('comment_id').loc[model_df.comment_id, 'cluster']).to_numpy(dtype=int)
        if not np.array_equal(final_model.predict(Z), expected_labels) or not np.array_equal(final_model.labels_, expected_labels):
            raise ValueError('Frozen assignments do not match these rows; topic names are not safe to apply.')
        stability = pd.read_csv(SAVED_MODEL_DIR / 'initialization_stability.csv')
    else:
        stability = initialization_stability(Z, final_model, threads=THREADS, n_init=N_INIT)
    display(stability)
    print('Mean initialization ARI:', stability.adjusted_rand_index.mean())
    common_space_quality = pd.DataFrame([original_space_diagnostics(
        X, eval_indices, final_model.labels_, vectorizer,
    )])
    display(common_space_quality)
    distances = final_model.transform(Z)
    nearest_two = np.sort(distances, axis=1)[:, :2]
    assignment_margin = (nearest_two[:, 1] - nearest_two[:, 0]) / np.maximum(nearest_two[:, 1], 1e-12)
    model_df['assignment_margin'] = assignment_margin
    print('Share with relative centroid margin < 0.10:', float((assignment_margin < 0.10).mean()))

    boundary_diagnostics = model_df.groupby(final_model.labels_).agg(
        n_comments=('comment_id', 'size'), mean_margin=('assignment_margin', 'mean'),
        low_margin_share=('assignment_margin', lambda values: (values < 0.10).mean()),
    ).rename_axis('cluster').reset_index()
    display(boundary_diagnostics)


## Comparison with the fixed-50 baseline

Local comparison folders preserve the dimension grid and optional domain-stop-word experiments. With identical features, 50/100/200/400 components retain approximately 8.71%/14.41%/22.87%/34.80% variance. More retained variance did not consistently improve topic separation.

The selected PCA100/L2/k=18 configuration provides a recognizable AI-related cluster and interpretable local pay and experience groups. Inspected k=20 alternatives included generic-word clusters. Stronger vocabulary filtering reduced initialization stability and was not selected.
This is exploratory selection, not performance on an independent test set. Dimension, normalization, k and initialization count changed together; the complete difference cannot be attributed to PCA dimension alone.

Mixed clusters remain and overall separation is weak. A cluster mentioning `off` is not automatically a layoff topic. Bound suggestions apply only to the verified model; new models retain unreviewed labels.


In [ ]:
model_comparison = pd.DataFrame()
if features_ready:
    current = {
        'configuration': 'Selected PCA + row L2', 'components': actual_components,
        'retained_variance': retained, 'k': SELECTED_K,
        'largest_cluster_share': float(selected_diagnostics.largest_cluster_share.iloc[0]),
        'pca_space_silhouette': float(selected_diagnostics.silhouette.iloc[0]),
        'tfidf_cosine_silhouette': float(common_space_quality.tfidf_cosine_silhouette.iloc[0]),
        'mean_initialization_ari': float(stability.adjusted_rand_index.mean()),
        'n_init': N_INIT,
    }
    rows = [current]
    baseline_dir = RUN_DIR / 'topics_20261029T034634149868Z'
    if (baseline_dir / 'topic_pipeline.joblib').exists():
        previous = joblib.load(baseline_dir / 'topic_pipeline.joblib')
        previous_parameters = previous['parameters']
        expected_input = hashlib.sha256((RUN_DIR / 'comments_clean.csv').read_bytes()).hexdigest()
        if previous_parameters['input_sha256'] != expected_input:
            raise ValueError('Baseline corpus changed; cannot compare.')
        if not np.array_equal(previous['vectorizer'].get_feature_names_out(), vectorizer.get_feature_names_out()):
            raise ValueError('Baseline vocabulary differs; common-space comparison needs alignment.')
        previous_Z = previous['pca'].transform(X)
        previous_labels = previous['kmeans'].predict(previous_Z)
        old_quality = original_space_diagnostics(X, eval_indices, previous_labels, vectorizer)
        old_stability = pd.read_csv(baseline_dir / 'initialization_stability.csv')
        rows.insert(0, {
            'configuration': 'Original PCA50, no row L2', 'components': 50,
            'retained_variance': previous_parameters['explained_variance'],
            'k': previous['kmeans'].n_clusters,
            'largest_cluster_share': float(pd.Series(previous_labels).value_counts(normalize=True).max()),
            'pca_space_silhouette': previous_parameters['selected_silhouette'],
            'tfidf_cosine_silhouette': old_quality['tfidf_cosine_silhouette'],
            'mean_initialization_ari': float(old_stability.adjusted_rand_index.mean()),
            'n_init': previous_parameters['n_init'],
        })
    model_comparison = pd.DataFrame(rows)
    display(model_comparison)
    print('PCA-space scores use different geometries. Common TF-IDF cosine scores use identical rows.')


## 10. Topic suggestions and six analysis cases

Names use original-space TF-IDF means, distinctive terms, five nearest-centroid and five random comments per cluster. Confidence is an assistant qualitative judgment, pending independent human validation, not model accuracy.

The six requested cases are **C2, C5, C8, C9, C11 and C17**. Interpret them using semantics, sample size, project relevance and robustness; do not upgrade their confidence labels for inclusion.
The cases include rising, falling, phase-dependent and stable discussion and are not independently validated "best topics".
All 18 clusters remain in assignments and exports. The heatmap hides only C7 after computing all shares. Names and case IDs apply only to the verified frozen model.


In [ ]:
final_ready = False
topic_review = pd.DataFrame()
focus_ids = []
if features_ready and SELECTED_K is not None:
    model_df, topic_profiles, representative_comments = build_topic_profiles(
        X, Z, model_df, vectorizer, final_model, labels=TOPIC_LABELS, seed=SEED,
    )
    if review_binding_verified:
        topic_review = pd.DataFrame(TOPIC_REVIEW)
        topic_profiles = topic_profiles.drop(columns='topic_label').merge(topic_review, on='cluster', validate='one_to_one')
        topic_profiles['label_source'] = 'assistant_suggested'
        topic_profiles['human_validated'] = False
        for cluster, name in TOPIC_LABELS.items():
            topic_profiles.loc[topic_profiles.cluster.eq(cluster), ['topic_label', 'label_source', 'human_validated']] = [name, 'human_provided', True]
        focus_ids = CASE_STUDY_IDS.copy()
        assert set(focus_ids).issubset(set(topic_profiles.cluster))
    else:
        topic_profiles = topic_profiles.assign(semantic_confidence='Not assessed', employment_topic=False,
            interpretation_limit='New model: requires separate semantic review.', trend_judgment='Not assessed',
            label_source='unreviewed', human_validated=False)
    for field in ['semantic_evidence', 'project_relevance', 'robustness_assessment']:
        topic_profiles[field] = topic_profiles.cluster.map(
            lambda cluster: CASE_STUDY_CONTEXT.get(cluster, {}).get(field, '') if review_binding_verified else '')
    topic_profiles['focus_topic'] = topic_profiles.cluster.isin(focus_ids)
    topic_profiles['chart_label'] = 'C' + topic_profiles.cluster.astype(str) + ' ' + topic_profiles.topic_label
    topic_names = topic_profiles.set_index('cluster').topic_label.to_dict()
    chart_labels = topic_profiles.set_index('cluster').chart_label.to_dict()
    model_df['topic_label'] = model_df.cluster.map(topic_names)
    representative_comments = representative_comments.merge(
        topic_profiles[['cluster', 'topic_label', 'semantic_confidence', 'label_source', 'human_validated']],
        on='cluster', how='left', validate='many_to_one')
    boundary_diagnostics = boundary_diagnostics.merge(
        topic_profiles[['cluster', 'topic_label', 'semantic_confidence']], on='cluster', validate='one_to_one')
    display(topic_profiles[['cluster', 'topic_label', 'n_comments', 'share', 'semantic_confidence',
                            'employment_topic', 'focus_topic', 'trend_judgment', 'interpretation_limit']])
    display(representative_comments)
    fig, ax = plt.subplots(figsize=(14, 10))
    ordered = topic_profiles.sort_values('cluster')
    ax.barh(ordered.chart_label, ordered.n_comments,
            color=['#4c78a8' if k in focus_ids else '#b9c1ca' for k in ordered.cluster])
    ax.invert_yaxis()
    ax.set(title='Cluster sizes and suggested topics (blue: six analysis cases)', xlabel='Assigned comments')
    fig.tight_layout()
    figures['cluster_sizes'] = fig
    plt.show()
    final_ready = True


## 11. Two-dimensional view for illustration

Display PC1/PC2 from the raw PCA projection, with crosses at raw projected cluster means. Actual clustering can use normalized multidimensional coordinates.
This view does not reproduce the clustering distance. Its reported variance concerns only the first two raw principal components.


In [ ]:
if final_ready:
    fig, ax = plt.subplots(figsize=(15, 9))
    scatter = ax.scatter(Z_pca[:, 0], Z_pca[:, 1], c=final_model.labels_, cmap='tab20', s=8, alpha=0.45)
    centers_2d = np.array([Z_pca[final_model.labels_ == k, :2].mean(axis=0) for k in range(SELECTED_K)])
    ax.scatter(centers_2d[:, 0], centers_2d[:, 1], c='black', marker='x', s=60)
    ax.set(xlabel='PC1', ylabel='PC2', title=f'2D view of {pca.n_components_}D clusters ({pca.explained_variance_ratio_[:2].sum():.1%} variance)')
    colorbar = fig.colorbar(scatter, ax=ax, ticks=range(SELECTED_K), label='Cluster and suggested topic')
    colorbar.ax.set_yticklabels([chart_labels[k] for k in range(SELECTED_K)], fontsize=8)
    fig.tight_layout()
    figures['cluster_projection'] = fig
    plt.show()


## 12. Monthly shares and sensitivity; C7 hidden in the heatmap

Each topic share divides its monthly comment count by **all assigned comments across all 18 clusters**.
Equal-post shares give each sampled month-post pair total weight one. Leave-one-time-stratum-out ranges show sensitivity to dropping one time block; they are not confidence intervals.

Compare complete January-March and July-September periods. October is partial at the frozen cutoff of 2026-10-28 07:10:54 UTC, or 15:10:54 Singapore time. Partial months remain plotted with an asterisk/shading but are excluded from period comparisons. Missing whole months are not invented as zero activity.

The heatmap hides only C7. The other 17 rows retain their original values, so columns need not total 100%. Its shared linear scale uses the visible maximum, about 13.3%, rather than the C7-dominated maximum of about 33.4%.
Cells show actual percentages. There is no resampling, renormalization, row standardization or nonlinear color transformation. C12 remains visible.


In [ ]:
if final_ready:
    early_months = [f'{YEAR}-{month:02d}' for month in [1, 2, 3]]
    late_months = [f'{YEAR}-{month:02d}' for month in [7, 8, 9]]
    topic_counts, topic_shares, monthly_topics, trend_summary = topic_trend_diagnostics(
        model_df, coverage[['month', 'partial_month', 'source_window_complete']], early_months, late_months,
    )
    metadata_columns = ['cluster', 'topic_label', 'semantic_confidence', 'employment_topic', 'focus_topic', 'trend_judgment',
                        'semantic_evidence', 'project_relevance', 'robustness_assessment']
    monthly_topics = monthly_topics.merge(topic_profiles[metadata_columns], on='cluster', validate='many_to_one')
    trend_summary = trend_summary.merge(topic_profiles[metadata_columns], on='cluster', validate='one_to_one')
    focus_trend_summary = trend_summary.loc[trend_summary.focus_topic].copy()
    monthly_assignment_coverage = corpus.groupby('month').size().rename('sampled').to_frame()
    monthly_assignment_coverage['assigned'] = model_df.groupby('month').size()
    monthly_assignment_coverage['assignment_rate'] = monthly_assignment_coverage['assigned'] / monthly_assignment_coverage['sampled']
    display(monthly_assignment_coverage)
    display(trend_summary)
    assert np.allclose(topic_shares.sum(axis=1), 1)
    month_order = topic_shares.index.tolist()
    partial_months = set(coverage.loc[coverage.partial_month.astype(str).str.lower().eq('true'), 'month'])
    month_ticks = [month + ('*' if month in partial_months else '') for month in month_order]
    # Compute all 18 shares first, then hide C7 without rescaling.
    heatmap_shares = topic_shares.drop(columns=HEATMAP_HIDDEN_CLUSTERS, errors='ignore').copy()
    assert np.array_equal(heatmap_shares.to_numpy(), topic_shares[heatmap_shares.columns].to_numpy())
    heatmap_vmax = float(heatmap_shares.to_numpy().max())
    fig, ax = plt.subplots(figsize=(14, 11))
    chart = ax.imshow(heatmap_shares.T.to_numpy(), aspect='auto', vmin=0, vmax=heatmap_vmax, cmap='Blues')
    ax.set_xticks(range(len(month_order)), month_ticks, rotation=45, ha='right')
    ax.set_yticks(range(len(heatmap_shares.columns)), [chart_labels[k] for k in heatmap_shares.columns], fontsize=9)
    ax.set(xlabel='UTC month (* partial)', ylabel='Cluster and suggested topic', title='Topic shares: C7 hidden, original proportions retained')
    from matplotlib.ticker import PercentFormatter
    colorbar = fig.colorbar(chart, ax=ax, label='Share of all assigned comments (denominator includes C7)')
    colorbar.ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
    for row, cluster in enumerate(heatmap_shares.columns):
        for column, month in enumerate(month_order):
            value = float(heatmap_shares.loc[month, cluster])
            ax.text(column, row, f'{value * 100:.1f}%', ha='center', va='center', fontsize=7,
                    color='white' if value > heatmap_vmax * 0.57 else '#243447')
    fig.text(0.5, 0.01, 'Original shares retained; columns need not total 100%. Shared linear scale covers 17 visible clusters; no row standardization.',
             ha='center', fontsize=9)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    figures['monthly_heatmap'] = fig
    plt.show()


## 12.1 Six cases: semantics, size, relevance and robustness

The review table preserves original interpretation confidence and reports counts, overall shares, semantic evidence, project relevance and robustness judgments. Overall size includes January-October; quarterly comparisons use complete January-March and July-September only.
Qualitative explanations accompany actual measurements; no unvalidated composite score is constructed.

| Case | Analysis value | Complete-quarter shares and interpretation limits |
|---|---|---|
| C2 Manager/HR communication and resignation notice | Larger topic with an upward signal | 4.98% to 6.85%; equal-post weighting also rises. Mixed communication/notice wording does not establish worsening conflict. |
| C5 Pay, pay cuts and payment issues | Employment-related stable reference | 3.18% to 3.17%; equal-post shares are also stable. Do not show only changing topics. |
| C8 Job applications and opportunities | Core job-seeking discussion | 4.84% to 3.82%, with intermediate rebounds; not evidence that job seeking became easier. |
| C9 Job changes, new jobs and onboarding | Relatively coherent phase-dependent discussion | 2.95% to 3.30%; lower April-May, higher August-September. Discussion is not actual job-switching counts. |
| C11 Experience, education and qualifications | Credentials and job-fit discussion | 3.86% to 2.79%, with a June rebound; not evidence that employers lowered requirements. |
| C17 AI use, generated content and job substitution | AI-related work and content discussion | 2.57% to 1.55%; higher February-May, generally lower later. Only about 10-33 comments per complete month; not solely job-loss anxiety. |

The robustness table includes equal-post period changes, monthly counts, distinct-post counts, maximum single-post contribution and time-stratum sensitivity. These describe fixed assignments; they are not semantic accuracy, population confidence intervals or causal tests.

All six panels retain the original all-topic denominator. Blue denotes comment-weighted shares; orange denotes equal sampled-post weights. Bands show leave-one-stratum-out ranges; grey marks partial months. A shared zero-based y-axis supports scale comparisons.


In [ ]:
focus_case_review = pd.DataFrame()
focus_robustness = pd.DataFrame()
if final_ready and focus_ids:
    focus_case_review = topic_profiles.set_index('cluster').loc[focus_ids, [
        'topic_label', 'semantic_confidence', 'n_comments', 'share',
        'semantic_evidence', 'project_relevance', 'robustness_assessment', 'interpretation_limit',
    ]].reset_index()
    complete_rows = monthly_topics.loc[
        monthly_topics.cluster.isin(focus_ids)
        & ~monthly_topics.partial_month.astype(str).str.lower().eq('true')
    ]
    focus_robustness = complete_rows.groupby('cluster').agg(
        min_monthly_comments=('n_topic', 'min'), max_monthly_comments=('n_topic', 'max'),
        min_monthly_posts=('unique_posts', 'min'), max_monthly_posts=('unique_posts', 'max'),
        max_single_post_share=('top_post_share', 'max'),
    ).reset_index().merge(focus_trend_summary, on='cluster', validate='one_to_one')
    display(focus_case_review)
    display(focus_robustness[['cluster', 'topic_label', 'first3_share', 'last3_share', 'change_pp',
                            'equal_post_first3_mean', 'equal_post_last3_mean', 'equal_post_change_pp',
                            'min_monthly_comments', 'max_monthly_comments', 'min_monthly_posts',
                            'max_monthly_posts', 'max_single_post_share', 'trend_judgment']])
    fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True, sharey=True)
    x = np.arange(len(month_order))
    focus_max = monthly_topics.loc[monthly_topics.cluster.isin(focus_ids), ['topic_share', 'equal_post_share', 'maximum_share']].max().max()
    upper_percent = max(5.0, float(np.ceil(focus_max * 100 + 0.5)))
    for ax, cluster in zip(axes.ravel(), focus_ids):
        subset = monthly_topics.loc[monthly_topics.cluster.eq(cluster)].set_index('month').loc[month_order]
        profile = topic_profiles.set_index('cluster').loc[cluster]
        if subset.minimum_share.notna().any():
            ax.fill_between(x, subset.minimum_share.to_numpy() * 100, subset.maximum_share.to_numpy() * 100,
                            alpha=0.18, color='#4c78a8', label='Leave-one-time-stratum-out range')
        ax.plot(x, subset.topic_share.to_numpy() * 100, marker='o', color='#4c78a8', label='Comment-weighted share')
        ax.plot(x, subset.equal_post_share.to_numpy() * 100, marker='s', linestyle='--', color='#e8872d', label='Equal weight per sampled post')
        for index, month in enumerate(month_order):
            if month in partial_months:
                ax.axvspan(index - 0.5, index + 0.5, color='grey', alpha=0.15)
        ax.set_title(chart_labels[cluster] + '\n' + f'Interpretability: {profile.semantic_confidence}; sample share: {profile.share:.2%} (n={profile.n_comments})', fontsize=10)
        ax.set(ylabel='Share of all assigned comments (%)', ylim=(0, upper_percent))
        ax.set_xticks(x, [month[5:] + ('*' if month in partial_months else '') for month in month_order])
        ax.set_xlabel(f'{YEAR} UTC month (* partial)')
        ax.grid(axis='y', alpha=0.2)
        ax.legend(fontsize=8)
    fig.suptitle('Six topic cases: semantics, size, relevance and trend robustness\nShared y-axis and original shares; sensitivity ranges are not confidence intervals', fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    figures['focus_monthly_trends'] = fig
    plt.show()
else:
    print('No frozen-model case studies applied; new models require separate interpretation.')


## 13. Export auditable local results

Create a fresh timestamped folder with every input ID and topic status, vocabulary, PCA variance, k diagnostics, stability, named profiles, examples, monthly shares, six-case review, robustness, visible heatmap values, figures and the fitted pipeline.
Unassigned comments retain IDs and exclusion reasons. Export source hashes, label provenance, model reuse and visualization settings. Suggested names are not marked human-validated merely because they are displayed.
These Reddit data and model artifacts remain local and ignored by Git. Toxicity inference must keep its own training preprocessing.


In [ ]:
if final_ready and SAVE_TOPIC_OUTPUTS:
    output = RUN_DIR / (f'topics_named_pca{pca.n_components_}_' + ('l2_' if PCA_L2_NORMALIZE else 'raw_') + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    output.mkdir()
    assignments = model_df[['comment_id', 'cluster', 'topic_label', 'assignment_margin']].merge(
        topic_profiles[['cluster', 'semantic_confidence', 'label_source', 'human_validated', 'focus_topic']],
        on='cluster', validate='many_to_one')
    comment_topics = corpus[['comment_id', 'month']].merge(assignments, on='comment_id', how='left', validate='one_to_one')
    excluded_reasons = feature_excluded.set_index('comment_id')['reason']
    comment_topics['topic_status'] = comment_topics['comment_id'].map(excluded_reasons).fillna('assigned')
    vocabulary = pd.DataFrame({'term': vectorizer.get_feature_names_out(), 'idf': vectorizer.idf_})
    tables = {
        'topic_name_review.csv': topic_profiles, 'trend_summary.csv': trend_summary,
        'focus_trend_summary.csv': focus_trend_summary,
        'focus_case_review.csv': focus_case_review, 'focus_robustness.csv': focus_robustness,
        'heatmap_visible_shares.csv': heatmap_shares.rename_axis(index='month').reset_index(),
        'comment_topics.csv': comment_topics, 'topic_profiles.csv': topic_profiles,
        'monthly_topic_trends.csv': monthly_topics, 'excluded_from_modelling.csv': feature_excluded,
        'zero_tfidf_comments.csv': feature_excluded.loc[feature_excluded.reason.eq('zero_tfidf')],
        'representative_comments.csv': representative_comments, 'k_selection.csv': k_results,
        'model_comparison.csv': model_comparison, 'boundary_diagnostics.csv': boundary_diagnostics, 'common_space_quality.csv': common_space_quality, 'pca_variance.csv': pca_variance, 'pca_comparison.csv': pca_comparison, 'initialization_stability.csv': stability,
        'tfidf_vocabulary.csv': vocabulary, 'monthly_assignment_coverage.csv': monthly_assignment_coverage.reset_index(),
    }
    for name, table in tables.items():
        table.to_csv(output / name, index=False, encoding='utf-8-sig')
    for name, figure in figures.items():
        figure.savefig(output / (name + '.png'), dpi=150, bbox_inches='tight')
    parameters = {
        'seed': SEED, 'selected_k': SELECTED_K, 'k_values': K_VALUES, 'n_init': N_INIT,
        'silhouette_sample_size': len(eval_indices), 'pca_components': int(pca.n_components_),
        'pca_l2_normalize': PCA_L2_NORMALIZE, 'protected_phrases': [],
        'pca_solver': pca.svd_solver,
        'pca_whiten': False,
        'explained_variance': float(pca.explained_variance_ratio_.sum()),
        'feature_diagnostics': feature_diagnostics, 'pca_fit_seconds': pca_seconds,
        'selected_silhouette': float(selected_diagnostics.silhouette.iloc[0]),
        'versions': versions, 'topic_labels': topic_names,
        'model_reused': saved_bundle is not None, 'review_binding_verified': review_binding_verified,
        'review_source': str(SAVED_MODEL_DIR) if review_binding_verified else None,
        'review_binding': REVIEW_BINDING if review_binding_verified else None,
        'focus_rule': 'User-specified six cases: semantic interpretability, size, project relevance and trend robustness',
        'heatmap_hidden_clusters': HEATMAP_HIDDEN_CLUSTERS, 'heatmap_renormalized': False,
        'heatmap_color_scale': {'type': 'linear', 'vmin': 0.0, 'vmax': heatmap_vmax},
        'focus_topic_ids': focus_ids, 'label_source': 'assistant_suggested, pending human validation' if review_binding_verified else 'unreviewed',
        'all_labels_reviewed': bool(topic_profiles['label_reviewed'].all()),
        'tfidf': {'ngram_range': [1, 2], 'min_df': 3, 'max_df': 0.90,
                  'max_features': 10000, 'sublinear_tf': True, 'norm': 'l2',
                  'stop_words': stop_words, 'dtype': 'float32'},
        'exclude_moderation_notices': EXCLUDE_MODERATION_NOTICES,
        'source_manifest': manifest,
        'input_sha256': hashlib.sha256((RUN_DIR / 'comments_clean.csv').read_bytes()).hexdigest(),
    }
    joblib.dump({'vectorizer': vectorizer, 'pca': pca, 'normalizer': normalizer, 'kmeans': final_model,
                 'parameters': parameters}, output / 'topic_pipeline.joblib', compress=3)
    parameters['output_sha256'] = {name: hashlib.sha256((output / name).read_bytes()).hexdigest() for name in tables}
    (output / 'run_parameters.json').write_text(json.dumps(parameters, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Saved topic artifacts:', output)


## 14. Interpretation and limitations

Increasing components raises retained TF-IDF variance, not guaranteed semantic coverage. Row normalization reduces some size/length effects without guaranteeing one clear topic per comment.
Relative centroid margins indicate ambiguity, not classification probability. No low-margin comments are removed to improve monthly curves.

The 18 labels are assistant suggestions pending human review. They are bound to the verified assignments, and new fits do not inherit names by cluster ID.
The six requested cases preserve original confidence ratings: C2 rises, C5 is stable, C8/C11 show declining signals, C9 recovers after a middle-period low, and C17 shows a phase-dependent decline. This is exploratory case selection, not an independently validated ranking.

- **Hiding C7 changes the chart only.** All remaining values and denominators are preserved. The shared linear scale uses the 17 visible topics; column sums need not equal 100%. The model has not eliminated the mixed cluster.
- **Relative shares and small samples.** Fixed quotas do not estimate absolute subreddit volume. One topic's share can fall when others rise. C17 has about 10-33 and C9 about 13-38 comments per complete month; changes are not counts of real-world events.
- **Sampling and dependence.** Public-archive random-anchor candidates are not a simple random sample of a complete month. Comments within a post are related. Equal-post and time-block checks do not remove all source bias or provide population confidence intervals.
- **Model and label uncertainty.** Original TF-IDF cosine silhouette is about 0.0083 and initialization ARI about 0.60-0.68. C7+C12 still account for about 42.4% and lack a single theme. Qualitative labels need independent review.
- **Time and external validity.** October is partial and excluded from full-quarter comparisons. AI content identification and tool use are not identical to job-loss anxiety; no clear layoff cluster supports a direct layoffs.fyi indicator.

For the frozen model, C9 changes from about 2.95% to 3.30% and C17 from about 2.57% to 1.55% between the first and third quarters. Interpret these as sample discussion patterns, with the stated phase changes and limitations.

Reference: [scikit-learn text clustering example](https://scikit-learn.org/stable/auto_examples/text/plot_document_clustering.html). Its normalization example uses SVD; this project separately evaluates true centered PCA.
